# Sequence-Level 14-3-3 Binding Prediction Demo

This notebook shows how to:

- prepare an input DataFrame with sequence names and sequences
- choose a sequence-level threshold
- predict all `S/T` site probabilities for each sequence
- summarize the positive sites, their probabilities, and whether the sequence is predicted to bind 14-3-3


In [1]:
import pandas as pd

from features import predict_sequence_binding


/home/luvul/.conda/envs/1433predictor2026/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Example input

The input DataFrame should contain:

- `sequence_name`
- `sequence`


In [2]:
input_df = pd.DataFrame(
    {
        "sequence_name": [
            "protein_1",
            "protein_2",
        ],
        "sequence": [
            "ASAAAAAASAAAAAAT",
            "MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVD",
        ],
    }
)

input_df


,sequence_name,sequence
0,protein_1,ASAAAAAASAAAAAAT
1,protein_2,MSSQSHPDGLSGRDQPVELLNPARVNHMPSTVD


## Set sequence-level threshold

A sequence is predicted as 14-3-3 binding if the highest `S/T` site probability is greater than or equal to this threshold.


In [3]:
threshold = 0.58


## Helper function

This function runs sequence-level prediction for each row and returns one summary row per sequence.


In [4]:
def predict_sequences_from_dataframe(df, threshold):
    result_rows = []

    for _, row in df.iterrows():
        sequence_name = row["sequence_name"]
        sequence = row["sequence"]

        summary, site_results_df, _, _ = predict_sequence_binding(sequence, threshold=threshold)

        positive_sites = site_results_df.loc[
            site_results_df["predicted_positive"], "site"
        ].tolist()

        positive_site_probabilities_df = site_results_df.loc[
            site_results_df["predicted_positive"], ["site", "positive_probability"]
        ].copy()
        positive_site_probabilities_df["positive_probability"] = positive_site_probabilities_df[
            "positive_probability"
        ].round(6)

        positive_site_probabilities = [
            {
                "site": int(site_row["site"]),
                "probability": float(site_row["positive_probability"]),
            }
            for _, site_row in positive_site_probabilities_df.iterrows()
        ]

        result_rows.append(
            {
                "sequence_name": sequence_name,
                "positive_sites": positive_sites,
                "positive_site_probabilities": positive_site_probabilities,
                "sequence_binding_14_3_3": summary["predicted_positive"],
            }
        )

    return pd.DataFrame(result_rows)


## Run prediction


In [5]:
output_df = predict_sequences_from_dataframe(input_df, threshold=threshold)
output_df


,sequence_name,positive_sites,positive_site_probabilities,sequence_binding_14_3_3
0,protein_1,[],[],False
1,protein_2,[5],"[{'site': 5, 'probability': 0.68026}]",True


## Output columns

- `sequence_name`: input sequence name
- `positive_sites`: sites predicted as positive at the site level
- `positive_site_probabilities`: all positive sites and their site-level probabilities
- `sequence_binding_14_3_3`: sequence-level binding call based on the maximum site probability and threshold
